In [1]:
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any
from pydantic import BaseModel
from openpyxl import load_workbook


In [2]:

class unit_group(BaseModel):
    name: str
    count: int
    percent: float
    minor_group:str
    sub_major_group:str
    major_group:str

In [5]:
def read_data(file_path: str) -> (Dict[str, pd.DataFrame], List[str]):

    def load_sheets(file_path: str, sheet_names: List[str], header: int = 3) -> Dict[str, pd.DataFrame]:
        return {name: pd.read_excel(file_path, sheet_name=name, header=header) for name in sheet_names}
    
    sheet_names = pd.ExcelFile(file_path).sheet_names
    sheet_data = load_sheets(file_path, sheet_names[3:7])
    return sheet_data, sheet_names


In [6]:
file_path = '/Users/leonardhaas/code/streamlit/data/raw_data/Zensus22_Sonderauswertung_Haas.xlsx'

sheet_data,sheet_names = read_data(file_path)   
haupt_gruppen_1 = sheet_data[sheet_names[3]]
berufs_gruppen_2 = sheet_data[sheet_names[4]]
berufs_unter_gruppen_3 = sheet_data[sheet_names[5]]
berufs_gattungen_4 = sheet_data[sheet_names[6]]

In [7]:
haupt_gruppen_1

,ISCO-Code,Bezeichnung,Insgesamt,Baden-Württemberg,Bayern,Berlin,Brandenburg,Bremen,Hamburg,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Nordrhein-Westfalen,Rheinland-Pfalz,Saarland,Sachsen,Sachsen-Anhalt,Schleswig-Holstein,Thüringen
0,Insgesamt,Insgesamt,41043450,5667810,7024330,1772180,1208030,321090,948980,3048160,723350,3935110,8621290,2025940,470130,1837640,971950,1479240,988230
1,0,Angehörige der regulären Streitkräfte,151210,10580,21860,3610,5960,1240,2170,7310,7210,21980,28190,11370,1850,5660,5510,11950,4750
2,1,Führungskräfte,1877040,266490,336750,95640,56450,13720,55550,149810,28770,159480,384110,83850,16940,82990,37060,68230,41180
3,2,Akademische Berufe,9039670,1308630,1549520,614410,231210,77070,307190,731260,120370,757830,1863260,390230,91920,382580,154550,282830,176830
4,3,Techniker und gleichrangige nichttechnische Be...,8777610,1201390,1478760,359840,269010,60700,201540,649620,151480,833590,1893940,436070,104570,398580,205060,322460,210980
5,4,Bürokräfte und verwandte Berufe,4883780,659850,875450,166060,140110,37710,108040,393500,85990,468950,1031460,251240,55170,201360,115760,182980,110150
6,5,Dienstleistungsberufe und Verkäufer,5758410,707890,951990,247140,187470,45710,122300,415450,129680,580440,1215110,293010,68870,266860,146360,243620,136510
7,6,Fachkräfte in Land-/Forstwirtschaft und Fischerei,554230,65440,106800,9900,20770,2150,5000,30540,15370,79590,99090,32110,4490,23470,15500,31590,12420
8,7,Handwerks- und verwandte Berufe,4707970,706170,856620,117860,144920,31970,62860,291580,88550,476620,936840,233660,59330,253830,138060,158520,150590
9,8,Bediener von Anlagen/Maschinen und Montageberufe,2297230,301600,385780,58240,74360,17970,35320,150150,44000,234860,493430,129600,28610,116550,78950,71580,76230


In [46]:
bundesländer_cols =['Baden-Württemberg', 'Bayern',
       'Berlin', 'Brandenburg', 'Bremen', 'Hamburg', 'Hessen',
       'Mecklenburg-Vorpommern', 'Niedersachsen', 'Nordrhein-Westfalen',
       'Rheinland-Pfalz', 'Saarland', 'Sachsen', 'Sachsen-Anhalt',
       'Schleswig-Holstein', 'Thüringen']

In [68]:
def clean_group(df: pd.DataFrame,bundesländer_cols:list) -> pd.DataFrame:
   # Replace '/' with '0' and convert 'Insgesamt' column to integer
   df = df.replace('/', '0')
   df['Insgesamt'] = df['Insgesamt'].astype(int)
   
   # Calculate the total and percentage
   total = df['Insgesamt'].iloc[0]
   df['percent'] = (df['Insgesamt'] / total) * 100
   
   # Drop unnecessary columns and the first row
   df = df.drop(columns=bundesländer_cols)
   df = df.drop(index=0)
   
   return df


In [69]:
cleaned_berufsgruppe2 = clean_group(berufs_gruppen_2, bundesländer_cols)

In [70]:
cleaned_berufsgruppe2.dtypes

ISCO-Code       object
Bezeichnung     object
Insgesamt        int64
percent        float64
dtype: object